# Bayesian Layer — Likelihood Ratio (LR) Reference Table

This notebook documents all LR values used in the HalfFull Bayesian update layer (`bayesian/lr_tables.json`).

**How LRs are used:**  
For each triggered condition (ML prior > 0.40), the user answers one structured question. The answer's LR updates the prior via odds-form Bayes:
```
posterior_odds = prior_odds × LR(answer)
posterior_prob = posterior_odds / (1 + posterior_odds)  [clipped to 0.05–0.95]
```

**Confidence tiers:**
- ✅ **High** — LR from meta-analysis or multi-centre validation study
- ⚠️ **Medium** — LR from single well-designed study or derived from sens/spec
- 🔴 **Low** — Clinical estimate or source does not directly validate LR — treat as placeholder

**PHQ-2 / GAD-2 confounders** are asked when any condition exceeds the 0.40 threshold. A positive screen downgrades all physical condition priors via a global multiplier.

In [ ]:
import json
import pandas as pd
from IPython.display import display, HTML

with open('../bayesian/lr_tables.json') as f:
    tables = json.load(f)

CONF_ICON = {'high': '✅ High', 'medium': '⚠️ Medium', 'low': '🔴 Low'}

def build_rows(conditions: dict) -> list[dict]:
    rows = []
    for cond_name, cond_data in conditions.items():
        for q in cond_data.get('questions', []):
            gender = q.get('gender_filter', '')
            gender_tag = ' ♀' if gender == 'female' else ''
            for opt in q.get('answer_options', []):
                lr = opt.get('lr')
                if lr is None:
                    continue
                conf = opt.get('lr_confidence', '?')
                source = opt.get('lr_source', opt.get('lr_note', ''))
                # Truncate source for display
                source_short = source[:90] + '…' if len(source) > 90 else source
                rows.append({
                    'Condition': cond_name,
                    'Question (patient-facing)': q['text'][:80] + ('…' if len(q['text']) > 80 else '') + gender_tag,
                    'Answer': opt['label'],
                    'LR': lr,
                    'Confidence': CONF_ICON.get(conf, conf),
                    'Source': source_short,
                })
    return rows

df = pd.DataFrame(build_rows(tables['conditions']))
print(f'{len(df)} rows across {df["Condition"].nunique()} conditions')
df.head(3)

## Full LR table — all answers

In [ ]:
all_answers = df.sort_values(['Condition', 'Question (patient-facing)', 'LR'], ascending=[True, True, False])

# Colour rows by confidence
def colour_conf(val):
    if 'High' in str(val):   return 'background-color: #d4edda'
    if 'Medium' in str(val): return 'background-color: #fff3cd'
    if 'Low' in str(val):    return 'background-color: #f8d7da'
    return ''

# Also shade LR values: green > 1, red < 1
def colour_lr(val):
    try:
        v = float(val)
        if v > 1:   return 'color: #155724; font-weight: bold'
        if v < 1:   return 'color: #721c24'
        return ''
    except Exception:
        return ''

styled = (
    all_answers
    .style
    .applymap(colour_conf, subset=['Confidence'])
    .applymap(colour_lr, subset=['LR'])
    .format({'LR': '{:.2f}'})
    .set_properties(**{'text-align': 'left', 'font-size': '12px'})
    .hide(axis='index')
)
display(styled)

## Summary: best question per condition (highest LR+)

In [ ]:
best = (
    df[df['LR'] > 1]
    .sort_values('LR', ascending=False)
    .groupby('Condition', sort=False)
    .first()
    .reset_index()
    .sort_values('LR', ascending=False)
    [['Condition', 'Question (patient-facing)', 'Answer', 'LR', 'Confidence', 'Source']]
)

styled_best = (
    best
    .style
    .applymap(colour_conf, subset=['Confidence'])
    .applymap(colour_lr, subset=['LR'])
    .format({'LR': '{:.2f}'})
    .set_properties(**{'text-align': 'left', 'font-size': '12px'})
    .hide(axis='index')
)
display(styled_best)

## Confidence breakdown

In [ ]:
summary = (
    all_answers
    .groupby(['Condition', 'Confidence'])
    .size()
    .unstack(fill_value=0)
)

total = all_answers.groupby('Condition').size().rename('Total answers')
summary = summary.join(total)
display(summary)

## PHQ-2 / GAD-2 confounder scoring

In [ ]:
conf_rows = []
for name, data in tables['confounders'].items():
    if name.startswith('_'):
        continue
    for band in data['scoring']['thresholds']:
        conf_rows.append({
            'Instrument': data['instrument'],
            'Score range': f"{band['score_range'][0]}–{band['score_range'][1]}",
            'Label': band['label'],
            'Multiplier on physical condition priors': band['lr_physical_conditions'],
            'Confidence': CONF_ICON.get(band.get('lr_confidence', ''), ''),
            'Source': band.get('lr_source', '')[:90],
        })

conf_df = pd.DataFrame(conf_rows)
display(
    conf_df.style
    .applymap(colour_conf, subset=['Confidence'])
    .hide(axis='index')
)

## Raw JSON — full lr_tables.json

---
## Testing the Bayesian layer

Three levels of testing — run them in order.

### Level 1 · Python subprocess (no server needed)
Pipe JSON directly to `run_bayesian.py`, exactly as Next.js does.  
Run the two cells below from this notebook, or paste the `echo | python3` commands into a terminal from the project root.

### Level 2 · Notebook (interactive, iterate fast)
The cells after Level 1 call the subprocess from Python so you can tweak inputs and inspect outputs inline.

### Level 3 · API end-to-end
Requires `cd frontend && npm run dev` in a terminal first, then run the curl cell at the bottom.

### Level 1 — terminal commands (copy/paste from project root)

In [ ]:
# Level 1 — equivalent terminal commands for reference (not executed here)
print("""
# From project root: halfFull/

# 1a. Questions mode — which questions fire for these ML scores?
echo '{
  "mode": "questions",
  "ml_scores": { "thyroid": 0.62, "anemia": 0.55, "perimenopause": 0.71 },
  "patient_sex": "female"
}' | python3 bayesian/run_bayesian.py | python3 -m json.tool

# 1b. Update mode — compute posteriors from answers
echo '{
  "mode": "update",
  "ml_scores": { "thyroid": 0.62, "anemia": 0.55, "perimenopause": 0.71 },
  "patient_sex": "female",
  "confounder_answers": { "phq2_q1": 1, "phq2_q2": 0, "gad2_q1": 0, "gad2_q2": 0 },
  "answers_by_condition": {
    "thyroid":       { "thyroid_q3": "Yes" },
    "anemia":        { "anemia_q1": "Yes" },
    "perimenopause": { "peri_q2":   "Yes" }
  }
}' | python3 bayesian/run_bayesian.py | python3 -m json.tool
""")

### Level 2 — run subprocess from notebook and inspect output

Edit `TEST_ML_SCORES`, `TEST_ANSWERS`, and `TEST_CONFOUNDER_ANSWERS` to try different scenarios.

In [ ]:
import subprocess, json, sys

BAYESIAN_SCRIPT = '../bayesian/run_bayesian.py'

def run_bayesian(payload: dict) -> dict:
    result = subprocess.run(
        [sys.executable, BAYESIAN_SCRIPT],
        input=json.dumps(payload),
        capture_output=True,
        text=True,
    )
    if result.returncode != 0:
        raise RuntimeError(result.stderr)
    return json.loads(result.stdout)

# ── edit these to test different scenarios ──────────────────────────────────
TEST_ML_SCORES = {
    "thyroid":       0.62,
    "anemia":        0.55,
    "perimenopause": 0.71,
    "sleep_disorder": 0.30,   # below threshold — should NOT trigger questions
}
TEST_PATIENT_SEX = "female"

# Confounder answers: PHQ-2 (phq2_q1/q2) and GAD-2 (gad2_q1/q2), each 0–3
TEST_CONFOUNDER_ANSWERS = {
    "phq2_q1": 1,
    "phq2_q2": 0,
    "gad2_q1": 0,
    "gad2_q2": 0,
}

# One answer per triggered condition, keyed by question id
TEST_ANSWERS = {
    "thyroid":       {"thyroid_q3": "Yes"},   # dry/coarse skin → LR+ 5.0
    "anemia":        {"anemia_q1":  "Yes"},   # heavy periods  → LR+ 2.4
    "perimenopause": {"peri_q2":    "Yes"},   # hot flushes    → LR+ 3.1
}
# ────────────────────────────────────────────────────────────────────────────

print("✓ helpers loaded — run the next two cells")

In [ ]:
# 2a. Questions mode — what does the /clarify page receive?
q_result = run_bayesian({
    "mode":        "questions",
    "ml_scores":   TEST_ML_SCORES,
    "patient_sex": TEST_PATIENT_SEX,
})

print(f"Confounder questions ({len(q_result['confounder_questions'])}):")
for cq in q_result['confounder_questions']:
    print(f"  [{cq['id']}] {cq['text']}")

print(f"\nCondition questions ({len(q_result['condition_questions'])}):")
for cq in q_result['condition_questions']:
    q = cq['question']
    print(f"\n  {cq['condition'].upper()}  (ML prior = {cq['probability']:.0%})")
    print(f"  Q: {q['text']}")
    for opt in q['answer_options']:
        print(f"      [{opt['value']}]  {opt['label']}")

In [ ]:
# 2b. Update mode — prior → posterior with full LR chain
u_result = run_bayesian({
    "mode":                 "update",
    "ml_scores":            TEST_ML_SCORES,
    "patient_sex":          TEST_PATIENT_SEX,
    "confounder_answers":   TEST_CONFOUNDER_ANSWERS,
    "answers_by_condition": TEST_ANSWERS,
})

rows = []
for cond, detail in u_result["details"].items():
    if detail:
        prior     = detail.get("prior", float("nan"))
        posterior = detail.get("posterior", float("nan"))
        clipped   = detail.get("clipped", False)
        lrs       = " × ".join(
            f"LR({e['question_id']}={e['answer']})={e['lr']:.2f}"
            for e in detail.get("lrs_applied", [])
        ) or "—"
        delta = posterior - prior
        arrow = "↑" if delta > 0.01 else ("↓" if delta < -0.01 else "→")
        rows.append((cond, prior, posterior, arrow, clipped, lrs))
    else:
        # condition was below threshold — passed through unchanged
        rows.append((cond, TEST_ML_SCORES.get(cond, "?"), "—", "·", False, "not triggered"))

df_results = pd.DataFrame(rows, columns=["Condition", "Prior", "Posterior", "", "Clipped", "LR chain"])
df_results = df_results.sort_values("Prior", ascending=False)

def colour_delta(row):
    styles = [""] * len(row)
    try:
        p, po = float(row["Prior"]), float(row["Posterior"])
        col = "background-color: #d4edda" if po > p + 0.01 else (
              "background-color: #f8d7da" if po < p - 0.01 else "")
        styles[list(df_results.columns).index("Posterior")] = col
    except Exception:
        pass
    return styles

display(
    df_results.style
    .apply(colour_delta, axis=1)
    .format({"Prior": "{:.2f}", "Posterior": lambda v: f"{v:.2f}" if isinstance(v, float) else v})
    .hide(axis="index")
)

### Level 3 — full API end-to-end (requires dev server)

Start the server first in a terminal:
```bash
cd frontend && npm run dev
```
Then run the cell below.

In [ ]:
import urllib.request, urllib.error

BASE_URL = "http://localhost:3000"

def post_json(path: str, body: dict) -> dict:
    data = json.dumps(body).encode()
    req  = urllib.request.Request(
        BASE_URL + path,
        data=data,
        headers={"Content-Type": "application/json"},
        method="POST",
    )
    try:
        with urllib.request.urlopen(req, timeout=10) as resp:
            return json.loads(resp.read())
    except urllib.error.URLError as e:
        return {"error": str(e), "hint": "Is `npm run dev` running in frontend/?"}

# 3a. /api/bayesian-questions
q_api = post_json("/api/bayesian-questions", {
    "mlScores":   TEST_ML_SCORES,
    "patientSex": TEST_PATIENT_SEX,
})
print("=== /api/bayesian-questions ===")
if "error" in q_api:
    print("ERROR:", q_api)
else:
    print(f"  confounder_questions : {len(q_api.get('confounder_questions', []))}")
    print(f"  condition_questions  : {len(q_api.get('condition_questions', []))}")
    for cq in q_api.get("condition_questions", []):
        print(f"    {cq['condition']} → {cq['question']['id']}: {cq['question']['text'][:60]}…")

# 3b. /api/bayesian-update
u_api = post_json("/api/bayesian-update", {
    "mlScores":            TEST_ML_SCORES,
    "patientSex":          TEST_PATIENT_SEX,
    "confounderAnswers":   TEST_CONFOUNDER_ANSWERS,
    "answersByCondition":  TEST_ANSWERS,
})
print("\n=== /api/bayesian-update ===")
if "error" in u_api:
    print("ERROR:", u_api)
else:
    print("  posteriorScores:", u_api.get("posteriorScores", {}))

In [ ]:
print(json.dumps(tables, indent=2, ensure_ascii=False))